In [1]:
from google.colab import drive
import sqlite3
import shutil
import os
import pandas as pd

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = "/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization"

DRIVE_DB_PATH = os.path.join(
    PROJECT_ROOT,
    "Database",
    "demand_forecast.db"
)

LOCAL_DB_PATH = "/content/demand_forecast.db"

if os.path.exists(DRIVE_DB_PATH):
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB_PATH)
    print("Copied Notebook 02 database to local storage.")
else:
    raise FileNotFoundError(
        "Database not found. Complete Notebook 02 first."
    )

try:
    conn.close()
except:
    pass

conn = sqlite3.connect(LOCAL_DB_PATH)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

def checkpoint_to_drive():
    shutil.copy(LOCAL_DB_PATH, DRIVE_DB_PATH)
    print("Database copied back to Google Drive.")

print("\nWorking Database")
print(LOCAL_DB_PATH)

!pip -q install statsmodels xgboost

print("\nLibraries installed successfully.")

check = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM fact_daily_sales) AS daily_sales_rows,
    (SELECT COUNT(*) FROM fact_forecast_accuracy WHERE model_key = 1) AS naive_rows,
    (SELECT COUNT(*) FROM dim_item) AS item_rows
""", conn)

print("\nDatabase Validation")
print(check)

assert check["daily_sales_rows"][0] == 913000
assert check["naive_rows"][0] == 46000
assert check["item_rows"][0] == 50

print("\nNotebook 02 database loaded successfully.")

Mounted at /content/drive
Copied Notebook 02 database to local storage.

Working Database
/content/demand_forecast.db

Libraries installed successfully.

Database Validation
   daily_sales_rows  naive_rows  item_rows
0            913000       46000         50

Notebook 02 database loaded successfully.


In [3]:
schema = pd.read_sql_query("""
PRAGMA table_info(dim_date)
""", conn)

print(schema[["name", "type"]])

            name     type
0       date_key  INTEGER
1      full_date     TEXT
2           year  INTEGER
3        quarter  INTEGER
4          month  INTEGER
5     month_name     TEXT
6           week  INTEGER
7            day  INTEGER
8       day_name     TEXT
9    day_of_year  INTEGER
10    is_weekend  INTEGER
11    is_holiday  INTEGER
12  holiday_name     TEXT


In [4]:
base_data = pd.read_sql_query("""
SELECT
    f.store_key,
    f.item_key,
    f.date_key,
    f.sales_qty,
    CASE
        WHEN f.date_key BETWEEN 20171001 AND 20171231
        THEN 1
        ELSE 0
    END AS is_test_period,
    d.month,
    d.quarter,
    d.is_weekend,
    d.is_holiday
FROM fact_daily_sales f
JOIN dim_date d
ON f.date_key = d.date_key
ORDER BY
    f.store_key,
    f.item_key,
    f.date_key
""", conn)

print(f"Rows Loaded : {len(base_data):,}")
print(f"Columns     : {list(base_data.columns)}")

train_rows = (base_data["is_test_period"] == 0).sum()
test_rows = (base_data["is_test_period"] == 1).sum()

print(f"\nTrain Rows : {train_rows:,}")
print(f"Test Rows  : {test_rows:,}")

print("\nNull Values")
print(base_data.isnull().sum())

assert len(base_data) == 913000
assert test_rows == 46000
assert base_data.isnull().sum().sum() == 0

print("\nBase time series loaded and validated.")

Rows Loaded : 913,000
Columns     : ['store_key', 'item_key', 'date_key', 'sales_qty', 'is_test_period', 'month', 'quarter', 'is_weekend', 'is_holiday']

Train Rows : 867,000
Test Rows  : 46,000

Null Values
store_key         0
item_key          0
date_key          0
sales_qty         0
is_test_period    0
month             0
quarter           0
is_weekend        0
is_holiday        0
dtype: int64

Base time series loaded and validated.


In [5]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

sku_list = (
    base_data[["store_key", "item_key"]]
    .drop_duplicates()
    .values
    .tolist()
)

print(f"Fitting Holt-Winters for {len(sku_list)} SKUs...")

hw_results = []
hw_sku_mapes = []
hw_fail_log = []

t0 = time.time()

for i, (store_key, item_key) in enumerate(sku_list):

    series = base_data[
        (base_data.store_key == store_key) &
        (base_data.item_key == item_key)
    ].sort_values("date_key")

    train = series[series.is_test_period == 0]["sales_qty"].values

    test_rows = series[series.is_test_period == 1]

    test_actual = test_rows["sales_qty"].values
    test_dates = test_rows["date_key"].values

    try:

        model = ExponentialSmoothing(
            train,
            trend="add",
            damped_trend=True,
            seasonal="add",
            seasonal_periods=7,
            initialization_method="estimated"
        ).fit(optimized=True)

        forecast = np.maximum(
            model.forecast(len(test_actual)),
            0
        )

    except Exception as e:
        hw_fail_log.append((store_key, item_key, str(e)))
        continue

    errors = np.abs(test_actual - forecast)

    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.where(
            test_actual > 0,
            errors / test_actual,
            np.nan
        )

    hw_sku_mapes.append(np.nanmean(ape))

    for dk, act, fc, err, a in zip(
        test_dates,
        test_actual,
        forecast,
        errors,
        ape
    ):
        hw_results.append((
            int(dk),
            int(store_key),
            int(item_key),
            int(act),
            float(fc),
            float(err),
            None if np.isnan(a) else float(a),
            float(err ** 2)
        ))

    if (i + 1) % 100 == 0:
        print(
            f"{i+1}/{len(sku_list)} completed "
            f"({time.time()-t0:.0f}s)"
        )

print(f"\nTotal Time : {time.time()-t0:.0f} seconds")
print(f"Failures   : {len(hw_fail_log)}")

if hw_fail_log:
    print(hw_fail_log)

hw_headline_mape = np.mean(hw_sku_mapes) * 100

print(f"\nHeadline Holt-Winters MAPE : {hw_headline_mape:.1f}%")
print("Benchmark                 : 23.4%")

Fitting Holt-Winters for 500 SKUs...
100/500 completed (81s)
200/500 completed (141s)
300/500 completed (200s)
400/500 completed (259s)
500/500 completed (315s)

Total Time : 315 seconds
Failures   : 0

Headline Holt-Winters MAPE : 23.4%
Benchmark                 : 23.4%


In [6]:
cursor.execute("""
DELETE FROM fact_forecast_accuracy
WHERE model_key = 3
""")

conn.commit()

cursor.executemany("""
INSERT INTO fact_forecast_accuracy (
    date_key,
    store_key,
    item_key,
    model_key,
    actual_qty,
    forecast_qty,
    abs_error,
    abs_pct_error,
    squared_error
)
VALUES (?, ?, ?, 3, ?, ?, ?, ?, ?)
""", [
    (
        r[0],
        r[1],
        r[2],
        r[3],
        r[4],
        r[5],
        r[6],
        r[7]
    )
    for r in hw_results
])

conn.commit()

n_inserted = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_forecast_accuracy
WHERE model_key = 3
""", conn).iloc[0, 0]

print(f"Holt-Winters Rows Inserted : {n_inserted:,}")

db_mape = pd.read_sql_query("""
SELECT
AVG(sku_mape) * 100 AS headline_mape
FROM (
    SELECT
        store_key,
        item_key,
        AVG(abs_pct_error) AS sku_mape
    FROM fact_forecast_accuracy
    WHERE model_key = 3
      AND abs_pct_error IS NOT NULL
    GROUP BY
        store_key,
        item_key
)
""", conn).iloc[0, 0]

print(f"Database MAPE : {db_mape:.1f}%")
print(f"In-Memory MAPE: {hw_headline_mape:.1f}%")

assert n_inserted == 46000
assert abs(db_mape - hw_headline_mape) < 0.05

checkpoint_to_drive()

print("\nHolt-Winters results inserted, validated and checkpointed.")

Holt-Winters Rows Inserted : 46,000
Database MAPE : 23.4%
In-Memory MAPE: 23.4%
Database copied back to Google Drive.

Holt-Winters results inserted, validated and checkpointed.


In [7]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

already_done = pd.read_sql_query(
    """
    SELECT DISTINCT store_key, item_key
    FROM fact_forecast_accuracy
    WHERE model_key = 2
    """,
    conn
)

done_set = set(map(tuple, already_done.values.tolist()))

all_skus = (
    base_data[["store_key", "item_key"]]
    .drop_duplicates()
    .values
    .tolist()
)

remaining_skus = [
    s for s in all_skus
    if tuple(s) not in done_set
]

print(
    f"Total SKUs: {len(all_skus)}, "
    f"already done: {len(done_set)}, "
    f"remaining: {len(remaining_skus)}"
)

sarima_fail_log = []

t0 = time.time()

for i, (store_key, item_key) in enumerate(remaining_skus):

    series = base_data[
        (base_data.store_key == store_key) &
        (base_data.item_key == item_key)
    ].sort_values("date_key")

    train = series[
        series.is_test_period == 0
    ]["sales_qty"].values

    test_rows = series[
        series.is_test_period == 1
    ]

    test_actual = test_rows["sales_qty"].values
    test_dates = test_rows["date_key"].values

    try:
        model = SARIMAX(
            train,
            order=(1,1,1),
            seasonal_order=(1,1,1,7),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        forecast = np.maximum(
            model.forecast(len(test_actual)),
            0
        )

    except Exception as e:
        sarima_fail_log.append(
            (store_key, item_key, str(e))
        )
        continue

    errors = np.abs(test_actual - forecast)

    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.where(
            test_actual > 0,
            errors / test_actual,
            np.nan
        )

    rows = [
        (
            int(dk),
            int(store_key),
            int(item_key),
            2,
            int(act),
            float(fc),
            float(err),
            None if np.isnan(a) else float(a),
            float(err ** 2)
        )
        for dk, act, fc, err, a in zip(
            test_dates,
            test_actual,
            forecast,
            errors,
            ape
        )
    ]

    cursor.executemany(
        """
        INSERT INTO fact_forecast_accuracy
        (
            date_key,
            store_key,
            item_key,
            model_key,
            actual_qty,
            forecast_qty,
            abs_error,
            abs_pct_error,
            squared_error
        )
        VALUES (?,?,?,?,?,?,?,?,?)
        """,
        rows
    )

    if (i + 1) % 25 == 0:
        conn.commit()

    if (i + 1) % 100 == 0:
        conn.commit()
        checkpoint_to_drive()
        print(
            f"{i+1}/{len(remaining_skus)} completed "
            f"({time.time()-t0:.0f}s)"
        )

conn.commit()
checkpoint_to_drive()

n_total = pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT store_key || '-' || item_key) AS n
    FROM fact_forecast_accuracy
    WHERE model_key = 2
    """,
    conn
)["n"][0]

print(f"\nTotal time: {time.time()-t0:.0f}s")
print(f"SARIMA SKUs completed: {n_total}/500")
print(f"Failures: {len(sarima_fail_log)}")

if sarima_fail_log:
    print(sarima_fail_log)

Total SKUs: 500, already done: 0, remaining: 500
Database copied back to Google Drive.
100/500 completed (396s)
Database copied back to Google Drive.
200/500 completed (765s)
Database copied back to Google Drive.
300/500 completed (1147s)
Database copied back to Google Drive.
400/500 completed (1529s)
Database copied back to Google Drive.
500/500 completed (1895s)
Database copied back to Google Drive.

Total time: 1896s
SARIMA SKUs completed: 500/500
Failures: 0


In [10]:
n_sarima_rows = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n
    FROM fact_forecast_accuracy
    WHERE model_key = 2
    """,
    conn
).iloc[0, 0]

sarima_sku_mape = pd.read_sql_query("""
SELECT
    store_key,
    item_key,
    AVG(abs_pct_error) AS sku_mape
FROM fact_forecast_accuracy
WHERE model_key = 2
  AND abs_pct_error IS NOT NULL
GROUP BY
    store_key,
    item_key
""", conn)

n_skus = len(sarima_sku_mape)

sarima_headline_mape = (
    sarima_sku_mape["sku_mape"].mean() * 100
)

print(f"SARIMA rows in database: {n_sarima_rows} (expected 46,000)")
print(f"SKUs with a computed MAPE: {n_skus} (expected 500)")

print(
    f"Per-SKU MAPE -- min: {sarima_sku_mape['sku_mape'].min()*100:.1f}%, "
    f"median: {sarima_sku_mape['sku_mape'].median()*100:.1f}%, "
    f"max: {sarima_sku_mape['sku_mape'].max()*100:.1f}%"
)

print(f"\n*** SARIMA HEADLINE MAPE: {sarima_headline_mape:.1f}% ***")
print("(Benchmark from earlier validated run: 25.2%)")

assert n_sarima_rows == 46000
assert n_skus == 500
assert abs(sarima_headline_mape - 25.2) < 1.0

print("\nSARIMA validated against the benchmark.")
print(
    f"Running comparison so far: "
    f"Naive 19.9% | Holt-Winters 23.4% | SARIMA {sarima_headline_mape:.1f}%"
)

SARIMA rows in database: 46000 (expected 46,000)
SKUs with a computed MAPE: 500 (expected 500)
Per-SKU MAPE -- min: 14.3%, median: 24.6%, max: 42.7%

*** SARIMA HEADLINE MAPE: 25.2% ***
(Benchmark from earlier validated run: 25.2%)

SARIMA validated against the benchmark.
Running comparison so far: Naive 19.9% | Holt-Winters 23.4% | SARIMA 25.2%


In [11]:
import pandas as pd
import numpy as np

base_data["date_parsed"] = pd.to_datetime(
    base_data["date_key"],
    format="%Y%m%d"
)

base_data["day_of_week"] = (
    base_data["date_parsed"].dt.dayofweek
)

holiday_dates = pd.to_datetime(
    sorted(
        base_data.loc[
            base_data["is_holiday"] == 1,
            "date_parsed"
        ].unique()
    )
)

def nearest_holiday_distance(d):
    return int(
        np.min(
            np.abs((holiday_dates - d).days)
        )
    )

unique_dates = base_data["date_parsed"].unique()

distance_map = {
    d: nearest_holiday_distance(pd.Timestamp(d))
    for d in unique_dates
}

base_data["days_to_nearest_holiday"] = (
    base_data["date_parsed"].map(distance_map)
)

grp = base_data.groupby(
    ["store_key", "item_key"]
)["sales_qty"]

base_data["lag_1"] = grp.shift(1)
base_data["lag_7"] = grp.shift(7)
base_data["lag_14"] = grp.shift(14)
base_data["lag_28"] = grp.shift(28)

base_data["roll_mean_7"] = grp.transform(
    lambda x: x.shift(1).rolling(7).mean()
)

base_data["roll_std_7"] = grp.transform(
    lambda x: x.shift(1).rolling(7).std()
)

base_data["roll_mean_28"] = grp.transform(
    lambda x: x.shift(1).rolling(28).mean()
)

item_bands = pd.read_sql_query(
    """
    SELECT item_key, price_band
    FROM dim_item
    """,
    conn
)

base_data = base_data.merge(
    item_bands,
    on="item_key",
    how="left"
)

base_data["store_cat"] = (
    base_data["store_key"]
    .astype("category")
    .cat.codes
)

base_data["item_cat"] = (
    base_data["item_key"]
    .astype("category")
    .cat.codes
)

base_data["price_band_cat"] = (
    base_data["price_band"]
    .astype("category")
    .cat.codes
)

feature_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "roll_mean_7",
    "roll_std_7",
    "roll_mean_28",
    "day_of_week",
    "month",
    "quarter",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "store_cat",
    "item_cat",
    "price_band_cat"
]

model_df = base_data.dropna(
    subset=feature_cols
).copy()

train_df = model_df[
    model_df.is_test_period == 0
]

test_df = model_df[
    model_df.is_test_period == 1
]

print(f"Feature columns: {feature_cols}")

print(
    f"\nTrain rows: {len(train_df):,}"
)

print(
    f"Test rows: {len(test_df):,}"
)

print("\nNull check:")

print(
    train_df[feature_cols]
    .isnull()
    .sum()
    .sum()
)

print(
    test_df[feature_cols]
    .isnull()
    .sum()
    .sum()
)

assert len(test_df) == 46000

assert (
    train_df[feature_cols]
    .isnull()
    .sum()
    .sum()
    == 0
)

assert (
    test_df[feature_cols]
    .isnull()
    .sum()
    .sum()
    == 0
)

print("\nFeature engineering validated.")

Feature columns: ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_28', 'day_of_week', 'month', 'quarter', 'is_weekend', 'is_holiday', 'days_to_nearest_holiday', 'store_cat', 'item_cat', 'price_band_cat']

Train rows: 853,000
Test rows: 46,000

Null check:
0
0

Feature engineering validated.


In [12]:
import xgboost as xgb
import time

t0 = time.time()

xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(
    train_df[feature_cols],
    train_df["sales_qty"]
)

print(f"Trained in {time.time()-t0:.0f}s")

preds = np.maximum(
    xgb_model.predict(test_df[feature_cols]),
    0
)

test_df = test_df.copy()

test_df["forecast"] = preds

test_df["abs_error"] = np.abs(
    test_df["sales_qty"] - test_df["forecast"]
)

test_df["ape"] = np.where(
    test_df["sales_qty"] > 0,
    test_df["abs_error"] / test_df["sales_qty"],
    np.nan
)

xgb_sku_mape = (
    test_df
    .groupby(["store_key", "item_key"])["ape"]
    .mean()
)

xgb_headline_mape = (
    xgb_sku_mape.mean() * 100
)

print(
    f"\nSKUs evaluated: {xgb_sku_mape.notna().sum()} "
    "(expected 500)"
)

print(
    f"Per-SKU MAPE -- min: {xgb_sku_mape.min()*100:.1f}%, "
    f"median: {xgb_sku_mape.median()*100:.1f}%, "
    f"max: {xgb_sku_mape.max()*100:.1f}%"
)

print(
    f"\n*** XGBOOST HEADLINE MAPE: "
    f"{xgb_headline_mape:.1f}% ***"
)

print(
    "(Benchmark from earlier validated run: 13.3%)"
)

importances = pd.Series(
    xgb_model.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

print("\nTop 5 features by importance:")
print(importances.head(5))

assert xgb_sku_mape.notna().sum() == 500

assert abs(xgb_headline_mape - 13.3) < 1.0

print(
    "\nXGBoost result validated against benchmark."
)

Trained in 38s

SKUs evaluated: 500 (expected 500)
Per-SKU MAPE -- min: 7.2%, median: 12.4%, max: 28.3%

*** XGBOOST HEADLINE MAPE: 13.3% ***
(Benchmark from earlier validated run: 13.3%)

Top 5 features by importance:
roll_mean_7     0.487666
lag_7           0.268795
lag_14          0.068639
roll_mean_28    0.067469
is_weekend      0.038951
dtype: float32

XGBoost result validated against benchmark.


In [13]:
xgb_rows = list(zip(
    test_df["date_key"].astype(int),
    test_df["store_key"].astype(int),
    test_df["item_key"].astype(int),
    test_df["sales_qty"].astype(int),
    test_df["forecast"].astype(float),
    test_df["abs_error"].astype(float),
    [None if pd.isna(x) else float(x) for x in test_df["ape"]],
    (test_df["abs_error"] ** 2).astype(float)
))

cursor.execute("""
DELETE FROM fact_forecast_accuracy
WHERE model_key = 4
""")

conn.commit()

cursor.executemany("""
INSERT INTO fact_forecast_accuracy (
    date_key,
    store_key,
    item_key,
    model_key,
    actual_qty,
    forecast_qty,
    abs_error,
    abs_pct_error,
    squared_error
)
VALUES (?, ?, ?, 4, ?, ?, ?, ?, ?)
""", xgb_rows)

conn.commit()

n_inserted = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_forecast_accuracy
WHERE model_key = 4
""", conn).iloc[0,0]

print(f"XGBoost Rows Inserted : {n_inserted:,}")

db_mape = pd.read_sql_query("""
SELECT AVG(sku_mape) * 100 AS headline_mape
FROM (
    SELECT
        store_key,
        item_key,
        AVG(abs_pct_error) AS sku_mape
    FROM fact_forecast_accuracy
    WHERE model_key = 4
      AND abs_pct_error IS NOT NULL
    GROUP BY store_key, item_key
)
""", conn).iloc[0,0]

print(f"Database MAPE : {db_mape:.1f}%")
print(f"In-Memory MAPE: {xgb_headline_mape:.1f}%")

assert n_inserted == 46000
assert abs(db_mape - xgb_headline_mape) < 0.05

checkpoint_to_drive()

print("\nXGBoost results inserted, validated, and checkpointed.")

XGBoost Rows Inserted : 46,000
Database MAPE : 13.3%
In-Memory MAPE: 13.3%
Database copied back to Google Drive.

XGBoost results inserted, validated, and checkpointed.


In [14]:
all_models = pd.read_sql_query("""
SELECT
    fa.model_key,
    m.model_name,
    fa.store_key,
    fa.item_key,
    fa.abs_pct_error
FROM fact_forecast_accuracy fa
JOIN dim_model m
ON fa.model_key = m.model_key
WHERE fa.abs_pct_error IS NOT NULL
""", conn)


print("=== FINAL FOUR-MODEL COMPARISON ===\n")

comparison = []

for model_key, name in [
    (1, "Naive"),
    (2, "SARIMA"),
    (3, "Holt-Winters"),
    (4, "XGBoost")
]:
    sub = all_models[all_models.model_key == model_key]

    sku_mape = (
        sub.groupby(["store_key","item_key"])["abs_pct_error"]
        .mean()
    )

    mape_pct = sku_mape.mean() * 100

    print(
        f"{name:15s} MAPE = {mape_pct:5.1f}% "
        f"(n={len(sku_mape)} SKUs)"
    )

    comparison.append((name, mape_pct))


raw = pd.read_sql_query("""
SELECT
    model_key,
    store_key,
    item_key,
    actual_qty,
    forecast_qty
FROM fact_forecast_accuracy
""", conn)


raw["resid"] = raw["actual_qty"] - raw["forecast_qty"]


naive_sigma = (
    raw[raw.model_key == 1]
    .groupby(["store_key","item_key"])["resid"]
    .std()
    .rename("naive_sigma")
)


naive_mape = (
    all_models[all_models.model_key == 1]
    .groupby(["store_key","item_key"])["abs_pct_error"]
    .mean()
    .rename("naive_mape")
)


candidates = {}

for model_key, name in [
    (2,"SARIMA"),
    (3,"Holt-Winters"),
    (4,"XGBoost")
]:

    sigma = (
        raw[raw.model_key == model_key]
        .groupby(["store_key","item_key"])["resid"]
        .std()
        .rename(f"{name}_sigma")
    )

    mape = (
        all_models[all_models.model_key == model_key]
        .groupby(["store_key","item_key"])["abs_pct_error"]
        .mean()
        .rename(f"{name}_mape")
    )

    candidates[name] = pd.concat([mape, sigma], axis=1)


summary = pd.concat(
    [naive_mape, naive_sigma],
    axis=1
).reset_index()


mape_matrix = pd.concat(
    [
        candidates["SARIMA"]["SARIMA_mape"],
        candidates["Holt-Winters"]["Holt-Winters_mape"],
        candidates["XGBoost"]["XGBoost_mape"]
    ],
    axis=1
)


mape_matrix.columns = [
    "SARIMA",
    "Holt-Winters",
    "XGBoost"
]


best_model = mape_matrix.idxmin(axis=1)


summary["best_model_name"] = best_model.values

summary["best_mape"] = (
    mape_matrix.min(axis=1)
    .values
)


summary["best_sigma"] = [
    candidates[
        best_model.loc[idx]
    ][f"{best_model.loc[idx]}_sigma"].loc[idx]
    for idx in mape_matrix.index
]


summary["sigma_reduction_pct"] = (
    (1 - summary["best_sigma"] /
     summary["naive_sigma"]) * 100
)


print("\nPer-SKU best model distribution:")
print(summary["best_model_name"].value_counts())


print(
    f"\nPortfolio avg sigma reduction: "
    f"{summary['sigma_reduction_pct'].mean():.1f}%"
)


print(
    f"SKUs with best_sigma < naive_sigma: "
    f"{(summary['best_sigma'] < summary['naive_sigma']).sum()} / 500"
)


cursor.execute("""
DROP TABLE IF EXISTS sku_error_summary
""")


cursor.execute("""
CREATE TABLE sku_error_summary (
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    naive_mape REAL,
    naive_sigma REAL,
    best_model_name TEXT,
    best_mape REAL,
    best_sigma REAL,
    sigma_reduction_pct REAL,
    PRIMARY KEY(store_key,item_key)
)
""")


conn.commit()


cursor.executemany("""
INSERT INTO sku_error_summary
(
store_key,
item_key,
naive_mape,
naive_sigma,
best_model_name,
best_mape,
best_sigma,
sigma_reduction_pct
)
VALUES (?,?,?,?,?,?,?,?)
""",
summary[
[
"store_key",
"item_key",
"naive_mape",
"naive_sigma",
"best_model_name",
"best_mape",
"best_sigma",
"sigma_reduction_pct"
]
].itertuples(index=False, name=None)
)


conn.commit()


n_loaded = pd.read_sql_query(
"""
SELECT COUNT(*) AS n
FROM sku_error_summary
""",
conn
)["n"][0]


n_nulls = pd.read_sql_query(
"""
SELECT COUNT(*) AS n
FROM sku_error_summary
WHERE naive_sigma IS NULL
OR best_sigma IS NULL
""",
conn
)["n"][0]


print(
f"\nsku_error_summary rows: {n_loaded}"
)

print(
f"Null sigma values: {n_nulls}"
)


assert n_loaded == 500
assert n_nulls == 0


checkpoint_to_drive()


print(
"\nsku_error_summary created and checkpointed to Drive."
)

=== FINAL FOUR-MODEL COMPARISON ===

Naive           MAPE =  19.9% (n=500 SKUs)
SARIMA          MAPE =  25.2% (n=500 SKUs)
Holt-Winters    MAPE =  23.4% (n=500 SKUs)
XGBoost         MAPE =  13.3% (n=500 SKUs)

Per-SKU best model distribution:
best_model_name
XGBoost    500
Name: count, dtype: int64

Portfolio avg sigma reduction: 32.7%
SKUs with best_sigma < naive_sigma: 500 / 500

sku_error_summary rows: 500
Null sigma values: 0
Database copied back to Google Drive.

sku_error_summary created and checkpointed to Drive.
